# div-prune on Colab (M1/M2 revision batches)

**Usage**: upload the whole `div-prune-colab` folder to Google Drive (`MyDrive/div-prune-colab`), then run the cells below in order.

- Do NOT upload the local `outputs/` folder: its CSV has the old column schema, which breaks the resume checks in B1/B3. A fresh `outputs/tables/results.csv` is created on Drive and accumulates there, so interrupted runs can resume across sessions. Download it afterwards and merge into the local paper CSV locally.
- `data/` is not needed either: PyG re-downloads Cora/CiteSeer/PubMed on Colab.

The three batch cells (B1/B2/B3) implement the IJPRAI revision plan:
- **B1** — M1 adaptive baselines (linear / kendall / gradnorm, 10 seeds, 3 datasets)
- **B2** — controller teacher checkpoints (best / mindiv / last) for M2; CSV rows go to a throwaway `outputs_ckpt` because the controller arm already exists in the paper CSV
- **B3** — M2 matrix: 4 criteria x 3 checkpoint sources x 10 seeds, reusing the B2 teachers (no dense retraining)

Each cell is resumable: it skips (dataset, seed, config) combos already present.

**Dropped-session recovery**: every finished combo is appended to `progress_log.md` on Drive with a timestamp (the last line is the exact breakpoint). After a disconnect: re-run cells 1-3 (env + Drive mount), then the recovery-check cell for a progress snapshot, then re-run the batch cell(s) — completed combos are skipped automatically. A B2 run interrupted mid-training simply retrains that (dataset, seed).

Key commands (adjust `experiment.runs`, `seed`, model/dataset as needed):
- Dense train: `python -m run.pipeline.train experiment=gat_cora experiment.runs=10 ...`
- Schedules: `experiment.lambda_schedule=fixed|controller|linear|kendall|gradnorm` + `experiment.regularizer=attention`
- Prune: `python -m run.pipeline.prune ... "+experiment.prune={criterion: diversity, ratio: 0.75, finetune_epochs: 100, checkpoint_source: late_min_div}"` (the `+` matters: `experiment.prune` is null in the yaml; string values containing `{}` must be inner-quoted, e.g. `reuse_stem: 'cora_seed{seed}_schedcontroller'`)

In [ ]:
# GPU check (expect a T4 or better)
!nvidia-smi

In [ ]:
# Install dependencies (torch is preinstalled on Colab)
!pip install -q torch-geometric hydra-core omegaconf tqdm scipy sacrebleu sentencepiece


In [ ]:
# Mount Drive, enter the uploaded project copy, then install divprune (editable)
import os
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive/div-prune-colab'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/div-prune-colab')
!pip install -q -e . --no-deps
!ls


In [ ]:
# === 断线恢复检查（幂等，可随时重跑） ===
# 断线/重连后：先重跑 cell 1-3（环境 + 挂载 + 进入目录），再跑本 cell 看进度快照，
# 然后直接重跑 B1/B2/B3 对应 cell——skip 逻辑会自动跳过已完成组合。
import os
from datetime import datetime

import pandas as pd

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"
df = pd.read_csv(csv_path) if os.path.exists(csv_path) else None
parts = []

if df is None or len(df) == 0:
    parts.append("CSV: none yet")
else:
    parts.append(f"CSV rows: {len(df)}")
    for sched in ["linear", "kendall", "gradnorm"]:
        b1 = df[(df.lambda_schedule == sched) & (df.prune_criterion.isna())]
        parts.append(f"B1 {sched}: {len(b1)}/30")
    pr = df[df.checkpoint_source.isin(["best", "late_min_div", "late_last"])]
    n_b3 = 0
    if len(pr):
        n_b3 = int(
            pr.groupby(["dataset", "prune_criterion", "checkpoint_source"])
            .seed.nunique()
            .clip(upper=10)
            .sum()
        )
    parts.append(f"B3: {n_b3}/360")

done_b2 = 0
for ds in ["cora", "citeseer", "pubmed"]:
    for seed in range(10):
        stem = f"{ds}_seed{seed}_schedcontroller"
        if all(
            os.path.exists(f"outputs_ckpt/checkpoints/{stem}_{s}.pt")
            for s in ("best", "mindiv", "last")
        ):
            done_b2 += 1
parts.append(f"B2 checkpoints: {done_b2}/30")

snap = " | ".join(parts)
plog(f"RECOVERY-CHECK: {snap}")
print("\nPROGRESS SNAPSHOT:\n  " + "\n  ".join(parts))

In [ ]:
# Quick smoke (2 seeds, 5 epochs) to verify the setup
!python -m run.pipeline.train experiment=gat_cora experiment.runs=2 model.epochs=5 model.hidden_per_head=8 experiment.regularizer=none seed=0

In [ ]:
# === B1: M1 adaptive baselines (linear / kendall / gradnorm), 10 seeds ===
# fixed + controller arms already exist in the paper CSV (B1/B5 protocol).
# Per-seed skip makes this cell resumable; every finished combo is appended to
# progress_log.md so a dropped session can be located from the last line.
import os
import subprocess
from datetime import datetime

import pandas as pd

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"
have = pd.read_csv(csv_path) if os.path.exists(csv_path) else None

def have_seed(ds, sched, seed):
    if have is None:
        return False
    m = (
        (have.dataset == ds)
        & (have.lambda_schedule == sched)
        & (have.seed == seed)
        & (have.prune_criterion.isna())
    )
    return m.any()

plog(f"START B1: csv_rows={0 if have is None else len(have)}")
for ds in ["cora", "citeseer", "pubmed"]:
    for sched in ["linear", "kendall", "gradnorm"]:
        for seed in range(10):
            if have_seed(ds, sched, seed):
                continue
            rc = run(
                f"python -m run.pipeline.train experiment=gat_cora "
                f"experiment.dataset={ds} experiment.lambda_schedule={sched} "
                f"experiment.regularizer=attention experiment.runs=1 seed={seed}"
            )
            plog(f"done rc={rc} B1 {ds} {sched} seed{seed}")
plog("END B1")

In [ ]:
# Inspect accumulated results: revision config coverage
import pandas as pd

df = pd.read_csv("outputs/tables/results.csv")
print("total rows:", len(df))
print(df.tail(8))

# B3 runs later in this notebook, so all-MISSING here is expected before B3 completes.
print("\nM2 coverage (seeds per criterion x checkpoint_source):")
pr = df[df.checkpoint_source.isin(["best", "late_min_div", "late_last"])]
if len(pr):
    cov = pr.groupby(["dataset", "prune_criterion", "checkpoint_source"]).seed.nunique()
    print(cov.unstack().to_string())
    missing = pr.groupby(["dataset", "prune_criterion", "checkpoint_source"]).seed.nunique()
    for (ds, crit, src), n in missing.items():
        if n < 10:
            print(f"MISSING: {ds} {crit} {src}: {n}/10 seeds")

print("\nB1 coverage (seeds per schedule):")
for sched in ["linear", "kendall", "gradnorm"]:
    b1 = df[(df.lambda_schedule == sched) & (df.prune_criterion.isna())]
    print(f"  {sched}: {len(b1)} rows, datasets={sorted(b1.dataset.unique())}")

In [ ]:
# === B2: controller teacher checkpoints for M2 (CSV goes to throwaway outputs_ckpt) ===
# The controller arm of M1 already exists in the paper CSV (target=0.7, eta=0.05,
# 10 seeds) - do NOT re-append those rows. We only need the rolling checkpoints
# (best / mindiv / last) that M2's checkpoint_source comparison loads.
# A (dataset, seed) is retrained iff any of its 3 checkpoints is missing, so a
# half-finished training run after a disconnect is simply redone. Progress lines
# go to progress_log.md (plog is redefined here so this cell works standalone).
import os
import subprocess
from datetime import datetime

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

plog("START B2")
for ds in ["cora", "citeseer", "pubmed"]:
    for seed in range(10):
        stem = f"{ds}_seed{seed}_schedcontroller"
        if all(
            os.path.exists(f"outputs_ckpt/checkpoints/{stem}_{s}.pt")
            for s in ("best", "mindiv", "last")
        ):
            continue
        rc = run(
            f"python -m run.pipeline.train experiment=gat_cora "
            f"experiment.dataset={ds} experiment.lambda_schedule=controller "
            f"experiment.regularizer=attention experiment.save_checkpoints=true "
            f"experiment.runs=1 seed={seed} output_dir=outputs_ckpt"
        )
        plog(f"done rc={rc} B2 {ds} seed{seed}")
plog("END B2")

In [ ]:
# === B3: M2 matrix (4 criteria x 3 checkpoint sources, 10 seeds) ===
# Adaptive (controller) teacher, ratio 0.75, FT 100 epochs - the M2 revision protocol.
# Each (dataset, seed) teacher checkpoint from B2 is shared across all 12 combos via
# reuse_stem + teacher_dir (no dense retraining). Rows append to outputs/tables/results.csv.
# Per-combo skip (re-reads the CSV on every call) makes this cell resumable; every
# finished combo is appended to progress_log.md so a dropped session can be located.
import os
import subprocess
from datetime import datetime

import pandas as pd

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"

def have_run(ds, crit, src, seed):
    # Re-read the CSV on every call so a partial rerun of this cell stays idempotent.
    if not os.path.exists(csv_path):
        return False
    have = pd.read_csv(csv_path)
    m = (
        (have.dataset == ds)
        & (have.prune_criterion == crit)
        & (have.checkpoint_source == src)
        & (have.seed == seed)
    )
    return m.any()

criteria = ["diversity", "magnitude", "gradient", "random"]
sources = ["best", "late_min_div", "late_last"]

plog("START B3")
for ds in ["cora", "citeseer", "pubmed"]:
    for crit in criteria:
        for src in sources:
            for seed in range(10):
                if have_run(ds, crit, src, seed):
                    continue
                # reuse_stem is a Python format template; {seed} must survive into the
                # Hydra override, so wrap it in inner single quotes.
                prune_cfg = (
                    "{criterion: " + crit + ", ratio: 0.75, finetune_epochs: 100, "
                    "checkpoint_source: " + src + ", "
                    "reuse_stem: '" + ds + "_seed{seed}_schedcontroller', "
                    "teacher_dir: outputs_ckpt/checkpoints}"
                )
                rc = run(
                    f"python -m run.pipeline.prune experiment=gat_cora "
                    f"experiment.dataset={ds} experiment.lambda_schedule=controller "
                    f"experiment.regularizer=attention experiment.runs=1 seed={seed} "
                    f'+experiment.prune="{prune_cfg}"'
                )
                plog(f"done rc={rc} B3 {ds} {crit} {src} seed{seed}")
plog("END B3")

In [ ]:
# B3 coverage check: 10 seeds per (dataset, criterion, checkpoint_source)
import pandas as pd

df = pd.read_csv("outputs/tables/results.csv")
pr = df[df.checkpoint_source.isin(["best", "late_min_div", "late_last"])]
if len(pr) == 0:
    print("No M2 rows yet - run the B3 cell above first.")
else:
    print("M2 coverage (seeds per criterion x checkpoint_source):")
    print(
        pr.groupby(["dataset", "prune_criterion", "checkpoint_source"])
        .seed.nunique()
        .unstack()
        .to_string()
    )
    ok = True
    for ds in ["cora", "citeseer", "pubmed"]:
        for crit in ["diversity", "magnitude", "gradient", "random"]:
            for src in ["best", "late_min_div", "late_last"]:
                n = pr[
                    (pr.dataset == ds)
                    & (pr.prune_criterion == crit)
                    & (pr.checkpoint_source == src)
                ].seed.nunique()
                if n < 10:
                    ok = False
                    print(f"MISSING: {ds} {crit} {src}: {n}/10 seeds")
    if ok:
        print("B3 complete: 360/360 (4 criteria x 3 sources x 3 datasets x 10 seeds).")

In [ ]:
# Inspect accumulated results
import pandas as pd
pd.read_csv('outputs/tables/results.csv').tail(20)

# Second revision (2026-09-07): B4 + H3 extension (Citeseer/Pubmed)

**Before running the cells below**, sync these two patched files from the
local repo to this Drive copy:

- `run/pipeline/train.py` — tolerates a ratio-only `experiment.prune` cfg
  (B4 from-scratch small models label the head budget without pruning).
- `run/pipeline/distill.py` — B5 rows now write `lambda_schedule` /
  `checkpoint_source` / `criterion_div` (new-protocol fields).

Batch plan (160 runs; each cell is resumable via per-seed skip on
`outputs/tables/results.csv`):
- B4: from-scratch GAT, 4/2 heads, no KD, no regularizer, 100 epochs,
  3 datasets x 2 ratios x 10 seeds.
- B5: pure-KD student from scratch (Citeseer/Pubmed only; Cora is already in
  the paper CSV), controller teacher trained in-memory, 100 epochs.
- B6: diversity prune + FT 100 epochs (Citeseer/Pubmed, ratio 0.5 only;
  ratio 0.75 is already covered by the B3 `best` rows).
- B8: diversity prune + KD 100 epochs (Citeseer/Pubmed), teacher = archived
  B2 checkpoint (`best` source) via `reuse_stem`.


In [ ]:
# === B4: from-scratch small models (4/2 heads, no KD), 10 seeds x 3 datasets ===
# num_heads = 8*(1-ratio): ratio 0.5 -> 4 heads, 0.75 -> 2 heads (matches the
# pruned-student structure). regularizer=none, distill=false, 100 epochs.
# Per-seed skip via outputs/tables/results.csv makes this cell resumable.
import os
import subprocess
from datetime import datetime

import pandas as pd

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"

def have_run(ds, ratio, seed):
    if not os.path.exists(csv_path):
        return False
    have = pd.read_csv(csv_path)
    m = (
        (have.dataset == ds)
        & (have.regularizer == "none")
        & (have.prune_ratio == ratio)
        & (have.distill == False)
        & (have.seed == seed)
    )
    return m.any()

plog("START B4")
for ds in ["cora", "citeseer", "pubmed"]:
    for ratio, heads in [(0.5, 4), (0.75, 2)]:
        for seed in range(10):
            if have_run(ds, ratio, seed):
                continue
            rc = run(
                f"python -m run.pipeline.train experiment=gat_cora "
                f"experiment.dataset={ds} experiment.regularizer=none "
                f"experiment.lambda_schedule=fixed experiment.runs=1 seed={seed} "
                f"model.num_heads={heads} model.epochs=100 "
                f"+experiment.prune.ratio={ratio}"
            )
            plog(f"done rc={rc} B4 {ds} ratio{ratio} seed{seed}")
plog("END B4")


In [ ]:
# === B5: pure-KD students from scratch, Citeseer + Pubmed, 10 seeds ===
# Teacher = adaptive dense controller trained in-memory (matches the Cora B5
# protocol in the paper). 100 epochs. Rows append to outputs/tables/results.csv.
import os
import subprocess
from datetime import datetime

import pandas as pd

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"

def have_run(ds, ratio, seed):
    if not os.path.exists(csv_path):
        return False
    have = pd.read_csv(csv_path)
    m = (
        (have.dataset == ds)
        & (have.distill == True)
        & (have.prune_criterion.isna())
        & (have.prune_ratio == ratio)
        & (have.seed == seed)
    )
    return m.any()

plog("START B5-h3ext")
for ds in ["citeseer", "pubmed"]:
    for ratio in [0.5, 0.75]:
        for seed in range(10):
            if have_run(ds, ratio, seed):
                continue
            rc = run(
                f"python -m run.pipeline.distill experiment=gat_cora "
                f"experiment.dataset={ds} experiment.lambda_schedule=controller "
                f"experiment.regularizer=attention experiment.runs=1 seed={seed} "
                f"+experiment.prune.ratio={ratio} +experiment.prune.finetune_epochs=100"
            )
            plog(f"done rc={rc} B5 {ds} ratio{ratio} seed{seed}")
plog("END B5-h3ext")


In [ ]:
# === B6 + B8: diversity prune + FT / + KD, Citeseer + Pubmed, 10 seeds ===
# Teacher: archived B2 controller checkpoint (best source) via reuse_stem, so no
# dense retraining. 100 FT epochs. Ratio 0.75 B6 rows are already covered by the
# B3 `best` rows (same protocol) and get skipped by have_run.
import os
import subprocess
from datetime import datetime

import pandas as pd

def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode

def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)

csv_path = "outputs/tables/results.csv"

def have_run(ds, ratio, kd, seed):
    if not os.path.exists(csv_path):
        return False
    have = pd.read_csv(csv_path)
    m = (
        (have.dataset == ds)
        & (have.prune_criterion == "diversity")
        & (have.prune_ratio == ratio)
        & (have.checkpoint_source == "best")
        & (have.distill == kd)
        & (have.seed == seed)
    )
    return m.any()

plog("START B6B8-h3ext")
for kd, label in [(False, "B6"), (True, "B8")]:
    for ds in ["citeseer", "pubmed"]:
        for ratio in [0.5, 0.75]:
            for seed in range(10):
                if have_run(ds, ratio, kd, seed):
                    continue
                prune_cfg = (
                    "{criterion: diversity, ratio: " + str(ratio) + ", finetune_epochs: 100, "
                    "checkpoint_source: best, "
                    "reuse_stem: '" + ds + "_seed{seed}_schedcontroller', "
                    "teacher_dir: outputs_ckpt/checkpoints}"
                )
                kd_flag = " experiment.distill=true" if kd else ""
                rc = run(
                    f"python -m run.pipeline.prune experiment=gat_cora "
                    f"experiment.dataset={ds} experiment.lambda_schedule=controller "
                    f"experiment.regularizer=attention experiment.runs=1 seed={seed}"
                    f"{kd_flag} +experiment.prune=\"{prune_cfg}\""
                )
                plog(f"done rc={rc} {label} {ds} ratio{ratio} seed{seed}")
plog("END B6B8-h3ext")


In [ ]:
# Coverage check: B4/B5/B6/B8 for the H3 extension (tab8_h3ext), 10 seeds each
import pandas as pd

df = pd.read_csv("outputs/tables/results.csv")
missing = []
for ds in ["cora", "citeseer", "pubmed"]:
    for ratio in [0.5, 0.75]:
        b4 = df[(df.dataset == ds) & (df.prune_ratio == ratio) &
                (df.regularizer == "none") & (df.distill == False)]
        b5 = df[(df.dataset == ds) & (df.prune_ratio == ratio) &
                (df.distill == True) & (df.prune_criterion.isna())]
        b6 = df[(df.dataset == ds) & (df.prune_ratio == ratio) &
                (df.distill == False) & (df.prune_criterion == "diversity")]
        b8 = df[(df.dataset == ds) & (df.prune_ratio == ratio) &
                (df.distill == True) & (df.prune_criterion == "diversity")]
        for label, sub in [("B4", b4), ("B5", b5), ("B6", b6), ("B8", b8)]:
            n_seeds = sub.seed.nunique()
            print(f"{ds} r{ratio} {label}: {n_seeds}/10")
            if n_seeds < 10:
                missing.append((ds, ratio, label))
print("MISSING:", missing if missing else "none - batch complete")
